# Guía: `pocketoption_connector` — funciones principales

Este notebook muestra **cómo usar** el módulo con ejemplos cortos.

**Aviso:** integración **no oficial** con Pocket Option; solo con fines educativos. Usa **cuenta demo** (`POCKETOPTION_DEMO=1`).

---

## Antes de ejecutar

1. Desde la **raíz del repo** (carpeta con `pyproject.toml`):
   - `pip install -e .`
   - Con login por email/contraseña: `pip install -e ".[credentials]"` y luego `playwright install chromium`
2. Copia `.env.example` → `.env` en la raíz y configura:
   - **Opción A:** `POCKETOPTION_SSID` (cookie del navegador), o
   - **Opción B:** `POCKETOPTION_EMAIL` y `POCKETOPTION_PASSWORD`
3. Si el login automático falla (CAPTCHA, 2FA), pon `POCKETOPTION_HEADLESS=0` en `.env`.

El símbolo de ejemplo `EURUSD_otc` debe existir en tu terminal web; cámbialo si hace falta.

In [ ]:
# Añade el proyecto al path si abriste el notebook sin instalar el paquete
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("Raíz del proyecto:", ROOT)

## 1. Import y constantes

La clase principal es **`PocketOption`**: API **síncrona** (sin `async`/`await`).

In [ ]:
from pocketoption_connector import PocketOption, resolve_ssid, __version__

ACTIVO = "EURUSD_otc"  # cámbialo por el par que uses en la web

print("Versión del paquete:", __version__)

## 2. Sesión con `PocketOption.session()`

Al entrar al `with`:
- Si existe **`POCKETOPTION_SSID`**, se usa (no abre navegador).
- Si no, intenta login con **email/contraseña** del `.env` (requiere Playwright).

Al salir del `with`, se **cierra** la conexión.

`load_dotenv=False` aquí porque ya cargamos `.env` arriba.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    print("Conectado. Objeto:", type(po).__name__)

## 3. Cuenta: `balance` y `account_snapshot()`

- **`po.balance`**: saldo (dict).
- **`po.account_snapshot()`**: saldo + datos extra de conexión si el backend los expone.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    print("--- balance ---")
    print(po.balance)
    print("--- snapshot ---")
    snap = po.account_snapshot()
    for clave in snap:
        print(clave, ":", type(snap[clave]).__name__)

## 4. Velas OHLC: `candles()` y `last_candle()`

- **`timeframe`**: duración de cada vela en **segundos** (ej. `60` = 1 minuto).
- **`count`**: cuántas velas pedir hacia el pasado.
- Si definiste `default_asset=` en la sesión, puedes omitir el primer argumento.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    velas = po.candles(timeframe=60, count=20)
    print(f"Recibidas {len(velas)} velas")
    if velas:
        print("Última:", velas[-1])
    print("Solo la más reciente:", po.last_candle(timeframe=60))

## 5. DataFrame (opcional)

Con `as_dataframe=True` necesitas **pandas** instalado. Si falla, usa el modo lista del ejemplo anterior.

In [ ]:
try:
    with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
        df = po.candles(timeframe=60, count=50, as_dataframe=True)
        print(df.tail(3))
except Exception as e:
    print("DataFrame no disponible:", e)

## 6. Polling ligero: `stream_candles()`

No es tiempo real tick-a-tick; repite `last_candle` cada `interval_seconds`. Útil para monitoreo simple.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    for i, vela in enumerate(
        po.stream_candles(timeframe=60, interval_seconds=3.0, max_updates=3)
    ):
        print(f"Actualización {i + 1}:", vela)

## 7. Listado de activos: `assets()`

Muchas versiones del cliente **no** devuelven lista; en ese caso usa los símbolos de la web.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    try:
        lista = po.assets()
        print("Primeros símbolos:", lista[:20])
    except Exception as exc:
        print("No hay API de listado o error:", type(exc).__name__, exc)

## 8. Conectar sin `with`: `from_env()` y `from_credentials()`

- **`PocketOption.from_env()`**: lee `.env` / variables, conecta y devuelve instancia ya abierta → **recuerda `po.close()`**.
- **`PocketOption.from_credentials(email, password, ...)`**: abre el navegador (Playwright), hace login y conecta.

In [ ]:
# Ejemplo from_env (usa SSID o email/password del .env)
po = PocketOption.from_env(default_asset=ACTIVO, load_dotenv=False)
try:
    print("Balance (from_env):", po.balance)
finally:
    po.close()

In [ ]:
# Descomenta y rellena solo si instalaste [credentials] y quieres probar login explícito
# po2 = PocketOption.from_credentials(
#     "tu@email.com",
#     "tu_contraseña",
#     headless=False,
#     default_asset=ACTIVO,
# )
# try:
#     print(po2.balance)
# finally:
#     po2.close()

## 9. Utilidad: `resolve_ssid()`

Obtiene el string SSID: primero desde `POCKETOPTION_SSID`, si no, login por navegador con email/contraseña del entorno.

In [ ]:
# Solo lectura del SSID resuelto (no imprimas el valor completo en público)
ssid = resolve_ssid(load_dotenv=False, prefer_env_ssid=True)
print("SSID resuelto, longitud:", len(ssid))

## 10. Operar (solo demo y bajo tu responsabilidad)

- **`buy_call` / `buy_put`**: `amount` en dinero de la cuenta; `duration` en segundos o `"1m"`, `"5m"`, etc.
- **`orders()`**: órdenes activas.
- **`check_order(id)`** y **`wait_order(id)`**: seguimiento del resultado (depende del backend).

**Descomenta** la celda siguiente solo en **cuenta demo**.

In [ ]:
# with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
#     orden = po.buy_call(amount=1.0, duration="1m")
#     print("Respuesta orden:", orden)
#     oid = orden.get("order_id") or orden.get("id")
#     if oid:
#         final = po.wait_order(oid, poll_seconds=2.0, timeout_seconds=120.0)
#         print("Estado final (si aplica):", final)
#     print("Activas:", po.orders())

## 11. Lo que **no** está en la API comunitaria

`po.tournaments()` y `po.signals()` lanzan error explicando que no están soportados en el transporte actual.

In [ ]:
with PocketOption.session(default_asset=ACTIVO, load_dotenv=False) as po:
    for nombre, fn in [("torneos", po.tournaments), ("señales", po.signals)]:
        try:
            fn()
        except Exception as exc:
            print(nombre, "→", type(exc).__name__)

## 12. API avanzada async: `PocketOptionConnector`

Si necesitas control fino con `async`/`await`, usa el conector de bajo nivel (mismo SSID que resuelve el flujo anterior).

In [ ]:
import asyncio

from pocketoption_connector import PocketOptionConnector, run_async, resolve_ssid

ssid = resolve_ssid(load_dotenv=False, prefer_env_ssid=True)

async def demo_async():
    conn = PocketOptionConnector(ssid, is_demo=True)
    await conn.connect()
    try:
        b = await conn.get_balance()
        print("Balance (async):", b)
    finally:
        await conn.disconnect()

run_async(demo_async())

---

### Resumen de funciones útiles

| Función / atributo | Qué hace |
|--------------------|----------|
| `PocketOption.session(...)` | Context manager: conecta / desconecta |
| `PocketOption.from_env()` | Conecta leyendo `.env` |
| `PocketOption.from_credentials()` | Login web automático (Playwright) |
| `po.balance` | Saldo |
| `po.account_snapshot()` | Saldo + extras |
| `po.candles(...)` | Historial OHLC |
| `po.last_candle()` | Última vela |
| `po.stream_candles(...)` | Polling de velas |
| `po.assets()` | Lista de símbolos (si existe) |
| `po.buy_call` / `po.buy_put` | Abrir operación |
| `po.orders()` | Órdenes activas |
| `po.check_order` / `po.wait_order` | Seguimiento |
| `resolve_ssid()` | Resolver SSID (env o navegador) |

Más detalle: `README.md` del repositorio.